In [ ]:
import pandas as pd
import numpy as np
import scipy
import sklearn
from scipy.stats import multivariate_normal 
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import discordance_scores as dd

In [ ]:
exp_matrix1 = pd.read_csv('/nethome/krangan7/hallou//exp_matrix1.csv')
exp_matrix2 = pd.read_csv('/nethome/krangan7/hallou//exp_matrix2.csv')
exp_matrix3 = pd.read_csv('/nethome/krangan7/hallou//exp_matrix3.csv')

In [ ]:
final_df = pd.read_csv('/nethome/krangan7/hallou/final_data_morph_gene.csv')

In [ ]:
final_df_1 = final_df[final_df['dataset_id'] == 'ds1']
final_df_2 = final_df[final_df['dataset_id'] == 'ds2'].reset_index(drop=True)
final_df_3 = final_df[final_df['dataset_id'] == 'ds3'].reset_index(drop=True)

In [ ]:
cells1 = final_df_1.index

genes_counts1 = exp_matrix1.loc[cells1].add_prefix('gene')
total_counts1 = genes_counts1.sum(axis=1)
keep1 = total_counts1 >= 20                                  # drop cells with too few molecules
counts1, total_counts1 = genes_counts1[keep1], total_counts1[keep1]

cells2 = final_df_2.index

genes_counts2 = exp_matrix2.loc[cells2].add_prefix('gene')
total_counts2 = genes_counts2.sum(axis=1)
keep2 = total_counts2 >= 20                                  # drop cells with too few molecules
counts2, total_counts2 = genes_counts2[keep2], total_counts2[keep2]

cells3 = final_df_3.index

genes_counts3 = exp_matrix3.loc[cells3].add_prefix('gene')
total_counts3 = genes_counts3.sum(axis=1)
keep3 = total_counts3 >= 20                                  # drop cells with too few molecules
counts3, total_counts3 = genes_counts3[keep3], total_counts3[keep3]


In [ ]:
E1 = final_df_1[final_df_1.filter(like='expr_').columns]
E2 = final_df_2[final_df_2.filter(like='expr_').columns]
E3 = final_df_3[final_df_3.filter(like='expr_').columns]

In [ ]:
M1 = final_df_1.filter(like='morph_').iloc[:, :20]
M2 = final_df_2.filter(like='morph_').iloc[:, :20]
M3 = final_df_3.filter(like='morph_').iloc[:, :20]

In [ ]:
cat1 = final_df_1[[ 'tm_celltype', 'tm_boundary_annotation']].astype(str)
groups1 = final_df_1['tm_spatial_block']

cat2 = final_df_2[[ 'tm_celltype', 'tm_boundary_annotation']].astype(str)
groups2 = final_df_2['tm_spatial_block']

cat3 = final_df_3[[ 'tm_celltype', 'tm_boundary_annotation']].astype(str)
groups3 = final_df_3['tm_spatial_block']

In [ ]:
gene_resids_1 = dd.gaussian_log_likelihood(E1, cat1, total_counts1, groups1)['resid']
gene_resids_2 = dd.gaussian_log_likelihood(E2, cat2, total_counts2, groups2)['resid']
gene_resids_3 = dd.gaussian_log_likelihood(E3, cat3, total_counts3, groups3)['resid']

In [ ]:
morph_resids_1 = dd.gaussian_log_likelihood(M1, cat1, total_counts1, groups1)['resid']
morph_resids_2 = dd.gaussian_log_likelihood(M2, cat2, total_counts2, groups2)['resid']
morph_resids_3 = dd.gaussian_log_likelihood(M3, cat3, total_counts3, groups3)['resid']

In [ ]:
gene_from_morph_resids1 = dd.ridge_regression(M1,E1,cat1,total_counts1, groups1)['resid']
gene_from_morph_resids2 = dd.ridge_regression(M2,E2,cat2,total_counts2, groups2)['resid']
gene_from_morph_resids3 = dd.ridge_regression(M3,E3,cat3,total_counts3, groups3)['resid']


In [ ]:
morph_from_gene_resids1 = dd.ridge_regression(E1,M1,cat1,total_counts1, groups1)['resid']
morph_from_gene_resids2 = dd.ridge_regression(E2,M2,cat2,total_counts2, groups2)['resid']
morph_from_gene_resids3 = dd.ridge_regression(E3,M3,cat3,total_counts3, groups3)['resid']


In [ ]:
a_x_1 = dd.nll_from_resid(gene_resids_1)
d_x_m_1 = dd.nll_from_resid(gene_from_morph_resids1)
a_m_1 = dd.nll_from_resid(morph_resids_1)
d_m_x_1 = dd.nll_from_resid(morph_from_gene_resids1)

a_x_2 = dd.nll_from_resid(gene_resids_2)
d_x_m_2 = dd.nll_from_resid(gene_from_morph_resids2)
a_m_2 = dd.nll_from_resid(morph_resids_2)
d_m_x_2 = dd.nll_from_resid(morph_from_gene_resids2)

a_x_3 = dd.nll_from_resid(gene_resids_3)
d_x_m_3 = dd.nll_from_resid(gene_from_morph_resids3)
a_m_3 = dd.nll_from_resid(morph_resids_3)
d_m_x_3 = dd.nll_from_resid(morph_from_gene_resids3)


In [ ]:
scores_1 = pd.DataFrame({
    'a_x':   a_x_1,
    'd_x_m': d_x_m_1,
    'a_m':   a_m_1,
    'd_m_x': d_m_x_1,
}, index=final_df_1.index)          # the same cells, in the same row order

scores_2 = pd.DataFrame({
    'a_x':   a_x_2,
    'd_x_m': d_x_m_2,
    'a_m':   a_m_2,
    'd_m_x': d_m_x_2,
}, index=final_df_2.index)          # the same cells, in the same row order

scores_3 = pd.DataFrame({
    'a_x':   a_x_3,
    'd_x_m': d_x_m_3,
    'a_m':   a_m_3,
    'd_m_x': d_m_x_3,
}, index=final_df_3.index)          # the same cells, in the same row order

In [ ]:
scores_1['DPair_1'] = 0.5 * (scores_1['d_x_m'] + scores_1['d_m_x'])
scores_2['DPair_2'] = 0.5 * (scores_2['d_x_m'] + scores_2['d_m_x'])
scores_3['DPair_3'] = 0.5 * (scores_3['d_x_m'] + scores_3['d_m_x'])

In [ ]:
def zscore_within(score, cat, min_group=10, robust=False):
    """score: Series indexed by cell; cat: DataFrame whose first two columns are cell type and boundary.
    Returns each cell's z-score relative to its own group. Groups smaller than min_group get NaN."""
    key = cat.iloc[:, 0].astype(str) + '|' + cat.iloc[:, 1].astype(str)
    g = score.groupby(key)
    if robust:
        center = g.transform('median')
        spread = g.transform(lambda x: 1.4826 * np.median(np.abs(x - np.median(x))))   # MAD scaled to match a standard deviation
    else:
        center, spread = g.transform('mean'), g.transform('std')
    z = (score - center) / spread
    return z.where(g.transform('size') >= min_group)

In [ ]:
scores_1 = pd.DataFrame({k: zscore_within(v, cat1) for k, v in
                       dict(AX=a_x_1, AM=a_m_1, DXgM=d_x_m_1, DMgX=d_m_x_1).items()})
scores_1.dropna(how='all', inplace=True)

scores_2 = pd.DataFrame({k: zscore_within(v, cat1) for k, v in
                       dict(AX=a_x_2, AM=a_m_2, DXgM=d_x_m_2, DMgX=d_m_x_2).items()})
scores_2.dropna(how='all', inplace=True)

scores_3 = pd.DataFrame({k: zscore_within(v, cat1) for k, v in
                       dict(AX=a_x_3, AM=a_m_3, DXgM=d_x_m_3, DMgX=d_m_x_3).items()})
scores_3.dropna(how='all', inplace=True)

In [ ]:
scores_1['DPair_1'] = 0.5 * (scores_1['DXgM'] + scores_1['DMgX'])
scores_2['DPair_2'] = 0.5 * (scores_2['DXgM'] + scores_2['DMgX'])
scores_3['DPair_3'] = 0.5 * (scores_3['DXgM'] + scores_3['DMgX'])

In [ ]:
scores_1

,AX,AM,DXgM,DMgX,DPair_1
0,0.065107,1.711086,0.038202,1.701380,0.869791
2,1.105744,-0.686822,1.291339,-0.580734,0.355303
3,-0.164675,0.339447,-0.152216,0.248242,0.048013
4,-0.738243,-0.778945,-0.817241,-0.740819,-0.779030
5,-0.343879,-0.086000,-0.308018,0.034956,-0.136531
...,...,...,...,...,...
1034,0.643899,-0.143203,0.567735,0.205430,0.386583
1035,0.811628,-1.035499,0.692215,-1.089431,-0.198608
1036,-1.114897,0.362207,-1.322430,0.271218,-0.525606
1037,0.554224,-0.228676,-0.169619,-0.622515,-0.396067


In [ ]:
from scipy.stats import quantile


a_x_1, a_m_1, d_1,d_mx,d_xm = scores_1['AX'] > quantile(scores_1['AX'], 0.9), scores_1['AM'] > quantile(scores_1['AM'], 0.9), scores_1['DPair_1'] > quantile(scores_1['DPair_1'], 0.9), scores_1['DMgX'] > quantile(scores_1['DMgX'], 0.9), scores_1['DXgM'] > quantile(scores_1['DXgM'], 0.9)

labels_1 = np.select(
    [ d_1, a_x_1 & ~a_m_1 & ~d_1, ~a_x_1 & a_m_1 & ~d_1, a_x_1 & a_m_1 & ~d_1, a_x_1 & a_m_1 & d_1],
    ['Discordance', 'Expression Outlier', 'Morphology Outlier', 'Both Abnormal, concordant', 'Both Abnormal, discordant'], default='Typical')

scores_1['category'] = np.where(scores_1[['AX', 'AM', 'DPair_1']].isna().any(axis=1), 'Not scored', labels_1)
print(scores_1['category'].value_counts())


a_x_2, a_m_2, d_2,d_mx_2,d_xm_2 = scores_2['AX'] > quantile(scores_2['AX'], 0.9), scores_2['AM'] > quantile(scores_2['AM'], 0.9), scores_2['DPair_2'] > quantile(scores_2['DPair_2'], 0.9), scores_2['DMgX'] > quantile(scores_2['DMgX'], 0.9), scores_2['DXgM'] > quantile(scores_2['DXgM'], 0.9)
labels_2 = np.select(
        [ d_2, a_x_2 & ~a_m_2 & ~d_2, ~a_x_2 & a_m_2 & ~d_2, a_x_2 & a_m_2 & ~d_2, a_x_2 & a_m_2 & d_2],
        ['Discordance', 'Expression Outlier', 'Morphology Outlier', 'Both Abnormal, concordant', 'Both Abnormal, discordant'], default='Typical')
scores_2['category'] = np.where(scores_2[['AX', 'AM', 'DPair_2']].isna().any(axis=1), 'Not scored', labels_2)
print(scores_2['category'].value_counts())



a_x_3, a_m_3, d_3,d_mx_3,d_xm_3 = scores_3['AX'] > quantile(scores_3['AX'], 0.9), scores_3['AM'] > quantile(scores_3['AM'], 0.9), scores_3['DPair_3'] > quantile(scores_3['DPair_3'], 0.9), scores_3['DMgX'] > quantile(scores_3['DMgX'], 0.9), scores_3['DXgM'] > quantile(scores_3['DXgM'], 0.9)
labels_3 = np.select(
        [ d_3, a_x_3 & ~a_m_3 & ~d_3, ~a_x_3 & a_m_3 & ~d_3, a_x_3 & a_m_3 & ~d_3, a_x_3 & a_m_3 & d_3],
        ['Discordance', 'Expression Outlier', 'Morphology Outlier', 'Both Abnormal, concordant', 'Both Abnormal, discordant'], default='Typical')
scores_3['category'] = np.where(scores_3[['AX', 'AM', 'DPair_3']].isna().any(axis=1), 'Not scored', labels_3)
print(scores_3['category'].value_counts())



category
Typical               812
Discordance           100
Morphology Outlier     46
Expression Outlier     41
Name: count, dtype: int64
category
Typical               675
Discordance            84
Morphology Outlier     43
Expression Outlier     33
Name: count, dtype: int64
category
Typical               681
Discordance            84
Morphology Outlier     41
Expression Outlier     35
Name: count, dtype: int64


In [ ]:
scores_1

,AX,AM,DXgM,DMgX,DPair_1,category
0,0.065107,1.711086,0.038202,1.701380,0.869791,Morphology Outlier
2,1.105744,-0.686822,1.291339,-0.580734,0.355303,Typical
3,-0.164675,0.339447,-0.152216,0.248242,0.048013,Typical
4,-0.738243,-0.778945,-0.817241,-0.740819,-0.779030,Typical
5,-0.343879,-0.086000,-0.308018,0.034956,-0.136531,Typical
...,...,...,...,...,...,...
1034,0.643899,-0.143203,0.567735,0.205430,0.386583,Typical
1035,0.811628,-1.035499,0.692215,-1.089431,-0.198608,Typical
1036,-1.114897,0.362207,-1.322430,0.271218,-0.525606,Typical
1037,0.554224,-0.228676,-0.169619,-0.622515,-0.396067,Typical


In [ ]:


def a_vs_a_scatter(df, q_d=0.90, q_a=0.90, ax=None, title=None,dataset_id=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5.5, 4.8), constrained_layout=True)
    else:
        fig = ax.figure

    hi_d = df['DPair_[{}]'.format(dataset_id)] >= df['DPair_[{}]'.format(dataset_id)].quantile(q_d)

    # background: all cells
    ax.scatter(df['AX'], df['AM'], s=6, color='#c8c3b8', alpha=0.7,
               linewidths=0, label='All cells', zorder=1)

    # foreground: high-discordance cells coloured by D_pair
    sc = ax.scatter(df.loc[hi_d, 'AX'], df.loc[hi_d, 'AM'],
                    c=df.loc[hi_d, 'DPair_[{}]'.format(dataset_id)], cmap='viridis', s=35,
                    edgecolor='#333333', linewidths=0.5, zorder=2)

    # abnormality cutoffs
    ax.axvline(df['AX'].quantile(q_a), color='#5a574f', ls='--', lw=1)
    ax.axhline(df['AM'].quantile(q_a), color='#5a574f', ls='--', lw=1)

    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label(r'Discordance ($D_{pair}$, z)')

    ax.set_xlabel(r'Expression abnormality ($A_X$, z)')
    ax.set_ylabel(r'Morphology abnormality ($A_M$, z)')
    if title: ax.set_title(title, loc='left')
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)

    # legend entries for the two layers
    from matplotlib.lines import Line2D
    ax.legend(handles=[
        Line2D([], [], marker='o', ls='', color='#c8c3b8', ms=5, label='All cells'),
        Line2D([], [], marker='o', ls='', color='#440154', mec='#333333', ms=7,
               label=f'High discordance (top {int((1-q_d)*100)}%)')],
        loc='upper right', frameon=False)
    return ax


In [ ]:
a_vs_a_scatter(scores_1, dataset_id=1, title='Dataset 1')
